# AgentFlight session runner

One Kaggle session = ONE phase. Flow (build-prompt section 9):
install -> start vLLM as subprocess -> run one phase CLI command -> push
`results/` to the HF dataset repo + commit ledger to GitHub.

**Session discipline:** never launch a phase whose single-task worst case
can't finish before the 9-hour cap; checkpoint after every task; phone-home
(push) after every phase.

In [ ]:
# 1. Bootstrap (installs pinned deps, ~5-10 min)
!bash kaggle/bootstrap.sh 2>&1 | tail -20

In [ ]:
# 2. Day-0 verification / E0 (skip if already PASS in this repo state)
!python experiments/e0_day0_check.py 2>&1 | tail -30
!python -m agentflight.cli budget --mark E0 --hours 1.0 --note "day-0 checklist"

In [ ]:
# 3. Run THE ONE phase this session is for. Examples:
# E1 recording (Mode-R input + P0 baseline):
!python -m agentflight.cli record --base-url http://127.0.0.1:8000/v1 --tier 4k --tag e1
# E2 replay matrix:
!python experiments/e2_replay.py --results results/
# E3 core policies:
!python -m agentflight.cli execute --policies P4,P5 --tag e3 --base-url http://127.0.0.1:8000/v1

In [ ]:
# 4. Validate + report + ledger + push (phone-home BEFORE session ends)
!python -m agentflight.cli check --results results/
!python -m agentflight.cli report --results results/ --out docs/results_report.md
!python -m agentflight.cli budget --mark E1 --hours 5.0 --note "session 1: E1 4k tier"
import os

os.environ.get('HF_TOKEN') and get_ipython().system('huggingface-cli upload agentflight/agentflight-results results/ --repo-type dataset')
get_ipython().system('git add results/ RESEARCH_LOG.md && git commit -m "results: session checkpoint" && git push')